# 10 · Does the terrain axis replicate on a second foundation model?

Every finding in this project so far is technically a claim about **Clay v1.5** specifically --
nothing has checked whether "PC1 is more fundamentally a terrain axis than a development axis"
(elevation r=-0.82) is a real property of satellite foundation models generally, or an artifact
of this one checkpoint. The paper's own limitations section says as much: this can't be ruled
out without a second model evaluated under identical conditions. This notebook is that check.

**Model:** [Prithvi-EO-2.0-300M](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M)
(IBM/NASA/Jülich), a independently-trained masked-autoencoder foundation model, loaded via
[terratorch](https://github.com/IBM/terratorch).

**A flag worth stating up front:** two independent lookups while building this notebook returned
inconsistent details about Prithvi's exact input bands and normalization stats (one source even
contradicted its own prose). Rather than trust either, `src/prithvi_embed.py` reads the real
bands/mean/std/image-size directly off the loaded model at runtime and prints them for a sanity
check before anything downstream uses them -- same defensive philosophy as `clay_embed.py`'s
metadata handling. **This notebook is more likely than most in this project to need a debugging
round on first run** -- if a cell errors, read the printed diagnostic (it's written to point at
the actual fix, not just "try again") before reporting back.

**Method:** refetch the exact same AOI and date range as notebook 01, chip it on Prithvi's own
reported image size (not assumed to be 224 like Clay's), embed every chip with Prithvi, then
rerun the two headline tests already established for Clay -- PCA-vs-elevation/NDVI correlation,
and KMeans-vs-WorldCover agreement -- entirely independently (own fetches, own NDVI computed from
Prithvi's own bands, not reused from `docs/data/`) so the comparison is fair. **Requires a GPU
runtime.**

In [ ]:
REPO_URL = "https://github.com/ZanderHirman08/SATEMB.git"

import os

if not os.path.exists("SATEMB"):
    !git clone {REPO_URL}
%cd SATEMB
!pip install -q -r environment/requirements-colab.txt
!pip install -q terratorch
# terratorch pulls in its own numpy/scipy pins, which can leave an installed
# numpy build that's binary-incompatible with the scipy already loaded in
# this Colab image (symptom: ImportError: cannot import name '_slice' from
# 'numpy._core.umath' the first time scipy is imported below). Forcing a
# fresh, mutually-consistent numpy+scipy pair after terratorch fixes the
# install; a plain pip upgrade doesn't help until the *process* restarts,
# since the old compiled extension is already loaded in memory.
!pip install -q --upgrade --force-reinstall numpy scipy
print("If this is the first time this cell has run in this session: go to")
print("Runtime -> Restart session now, then Run All again from the top.")
print("(Only needed once per fresh runtime -- installing packages doesn't")
print("reload already-imported C extensions in a live Python process.)")

In [ ]:
import sys

sys.path.append(os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
import odc.stac
import pandas as pd
import torch
from scipy import stats
from sklearn.metrics import adjusted_rand_score

from src import prithvi_embed, stac_utils, viz_utils

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    print("No GPU detected -- continuing on CPU. This re-embeds the full 725-chip grid,")
    print("so it will be considerably slower than notebook 08's small AOI.")

os.makedirs("docs/figures", exist_ok=True)
catalog = stac_utils.open_catalog()

## Load Prithvi and discover its real config

Don't trust hardcoded band/normalization assumptions -- read them off the loaded model.

In [ ]:
model = prithvi_embed.load_model(device=device)
cfg = prithvi_embed.discover_config(model)

BANDS = cfg["bands"]
MEAN = cfg["mean"]
STD = cfg["std"]
IMG_SIZE = int(cfg["img_size"])
print(f"\nWill fetch bands {BANDS}, chip at {IMG_SIZE}x{IMG_SIZE}px")

## Fetch the same AOI and date range as notebook 01

Same bbox, same date window, same per-tile least-cloudy selection -- but only the bands Prithvi
actually needs, and chipped at Prithvi's own reported image size rather than assuming 224px like
Clay's grid.

In [ ]:
from collections import defaultdict

items = stac_utils.search_sentinel2(catalog)
print(f"{len(items)} candidate scenes found")

by_tile = defaultdict(list)
for it in items:
    tile = it.properties.get("s2:mgrs_tile", "unknown")
    by_tile[tile].append(it)

mosaic_items = []
for tile, tile_items in sorted(by_tile.items()):
    best = min(tile_items, key=lambda it: it.properties.get("eo:cloud_cover", 100))
    mosaic_items.append(best)
    print(f"  tile {tile}: {best.id}  cloud_cover={best.properties.get('eo:cloud_cover'):.1f}%  date={best.datetime.date()}")

ds = odc.stac.load(
    mosaic_items, bands=BANDS, bbox=stac_utils.FRONT_RANGE_BBOX,
    crs="EPSG:32613", resolution=stac_utils.GSD_M, groupby="solar_day",
    chunks={"x": 1024, "y": 1024},
)
mosaic = ds.to_array(dim="band").median(dim="time").compute()
print(mosaic.shape, mosaic.dtype)

In [ ]:
height, width = mosaic.sizes["y"], mosaic.sizes["x"]
grid = stac_utils.make_pixel_chip_grid(height, width, chip_size_px=IMG_SIZE)
print(f"{len(grid)} candidate chips ({height // IMG_SIZE} rows x {width // IMG_SIZE} cols) at {IMG_SIZE}px")

x_coords, y_coords = mosaic.x.values, mosaic.y.values
raster_crs = ds.odc.crs
NODATA_FRAC_THRESHOLD = 0.05

chips_meta, chip_pixels = [], []
for chip in grid:
    arr = mosaic.isel(y=chip["y_slice"], x=chip["x_slice"]).values
    if np.isnan(arr).mean() > NODATA_FRAC_THRESHOLD:
        continue
    arr = np.nan_to_num(arr, nan=0.0).astype("float32")

    bounds = stac_utils.pixel_window_to_lonlat_bounds(x_coords, y_coords, chip, raster_crs)
    lat, lon = stac_utils.bounds_centroid(bounds)

    chips_meta.append({"id": chip["id"], "bounds": bounds, "lat": lat, "lon": lon,
                        "y_slice": chip["y_slice"], "x_slice": chip["x_slice"]})
    chip_pixels.append(arr)

chip_pixels = np.stack(chip_pixels)
print(f"Kept {len(chips_meta)}/{len(grid)} chips -> {chip_pixels.shape}")

## Embed every chip with Prithvi

In [ ]:
def embed_all(pixels, batch_size=16):
    out = []
    for start in range(0, len(pixels), batch_size):
        end = min(start + batch_size, len(pixels))
        batch_norm = prithvi_embed.normalize_chips(pixels[start:end], MEAN, STD)
        out.append(prithvi_embed.encode_batch(model, batch_norm, device=device))
        if device == "cuda":
            torch.cuda.empty_cache()
    return np.concatenate(out, axis=0)

embeddings = embed_all(chip_pixels)
print(f"Embedded {len(chips_meta)} chips, dim={embeddings.shape[1]}")

## PCA vs. elevation and NDVI

Independent fetches, not reused from `docs/data/` -- own USGS 3DEP elevation pull, own NDVI
computed directly from Prithvi's own fetched bands (whichever of them correspond to red and
near-infrared, found by name rather than assumed by position).

In [ ]:
dem_items = stac_utils.search_dem(catalog)
dem_ds = odc.stac.load(
    dem_items, bands=["data"], bbox=stac_utils.FRONT_RANGE_BBOX,
    crs="EPSG:32613", resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024},
)
dem_var = dem_ds["data"]
elevation_grid = dem_var.median(dim="time").compute().values if "time" in dem_var.dims else dem_var.compute().values
elevation_grid = np.where(elevation_grid < -1000, np.nan, elevation_grid)
print(f"Elevation grid: {elevation_grid.shape}")

mean_elev = np.array([
    float(np.nanmean(elevation_grid[m["y_slice"], m["x_slice"]])) for m in chips_meta
])
print(f"Matched elevation for {(~np.isnan(mean_elev)).sum()}/{len(chips_meta)} chips")

In [ ]:
def band_index(name_options):
    for name in name_options:
        if name in BANDS:
            return BANDS.index(name)
    raise ValueError(f"None of {name_options} found in Prithvi's bands {BANDS} -- can't compute NDVI.")

RED_IDX = band_index(["B04"])
NIR_IDX = band_index(["B8A", "B08", "B05"])  # narrow NIR, standard NIR, or red-edge as a fallback
print(f"NDVI from band index {RED_IDX} (red) and {NIR_IDX} (NIR)")

def chip_ndvi(pixels_chip):
    red = pixels_chip[RED_IDX].astype("float32")
    nir = pixels_chip[NIR_IDX].astype("float32")
    return float(np.mean((nir - red) / (nir + red + 1e-6)))

ndvi = np.array([chip_ndvi(chip_pixels[i]) for i in range(len(chips_meta))])
print(f"NDVI range: {ndvi.min():.2f} to {ndvi.max():.2f}")

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
pcs = pca.fit_transform(embeddings)
var_pct = pca.explained_variance_ratio_ * 100
print(f"Top-3 PCA components explain {var_pct.sum():.1f}% of variance ({', '.join(f'{v:.1f}%' for v in var_pct)})")

valid = ~np.isnan(mean_elev)
print("\nComponent | var% | r(elevation) | r(NDVI)")
best_r_elev, best_r_ndvi = 0.0, 0.0
for i in range(3):
    r_elev = stats.pearsonr(mean_elev[valid], pcs[valid, i])[0]
    r_ndvi = stats.pearsonr(ndvi, pcs[:, i])[0]
    print(f"PC{i+1}       | {var_pct[i]:.1f}% | {r_elev:+.3f}       | {r_ndvi:+.3f}")
    if abs(r_elev) > abs(best_r_elev):
        best_r_elev = r_elev
    if abs(r_ndvi) > abs(best_r_ndvi):
        best_r_ndvi = r_ndvi

print(f"\nStrongest PC-vs-elevation correlation found: r={best_r_elev:.3f}")
print("Compare against Clay's PC1-vs-elevation result: r=-0.82 (README / paper Table 4).")
print(f"Strongest PC-vs-NDVI correlation found: r={best_r_ndvi:.3f}")
print("Compare against Clay's PC1-vs-NDVI result: r=-0.55 (README / paper Table 2).")

In [ ]:
rgb = np.zeros((pcs.shape[0], 3))
for i in range(3):
    col = pcs[:, i]
    rgb[:, i] = (col - col.min()) / (col.max() - col.min() + 1e-9)

lats = [m["lat"] for m in chips_meta]
lons = [m["lon"] for m in chips_meta]

plt.figure(figsize=(9, 9))
plt.scatter(lons, lats, c=rgb, s=14, marker="s")
plt.gca().set_aspect(1 / np.cos(np.radians(np.mean(lats))))
plt.title("Prithvi embeddings: chips colored by PCA projection (RGB)")
plt.axis("off")
plt.savefig("docs/figures/prithvi_pca_semantic_map.png", dpi=150, bbox_inches="tight")
plt.show()

## KMeans clustering vs. ESA WorldCover

In [ ]:
WORLDCOVER_CLASSES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse", 70: "Snow/ice", 80: "Water",
    90: "Wetland", 95: "Mangroves", 100: "Moss/lichen",
}

wc_items = stac_utils.search_worldcover(catalog)
wc_ds = odc.stac.load(
    wc_items, bbox=stac_utils.FRONT_RANGE_BBOX, crs="EPSG:32613",
    resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024},
)
wc_var = wc_ds["map"]
wc_map = wc_var.isel(time=0).compute().values if "time" in wc_ds.dims else wc_var.compute().values

clusters = viz_utils.cluster_embeddings(embeddings, method="kmeans", n_clusters=8)
print(f"Cluster sizes: {np.bincount(clusters[clusters >= 0])}")

majority_class, keep_mask = [], []
for m in chips_meta:
    patch = wc_map[m["y_slice"], m["x_slice"]]
    values, counts = np.unique(patch[patch > 0], return_counts=True)
    if len(values) == 0:
        keep_mask.append(False)
        continue
    majority_class.append(values[np.argmax(counts)])
    keep_mask.append(True)

keep_mask = np.array(keep_mask)
majority_class = np.array(majority_class)
clusters_matched = clusters[keep_mask]
class_names = np.array([WORLDCOVER_CLASSES.get(c, str(c)) for c in majority_class])

ari = adjusted_rand_score(majority_class, clusters_matched)
print(f"\nMatched {keep_mask.sum()}/{len(chips_meta)} chips to a WorldCover majority class")
print(f"Prithvi ARI vs. WorldCover: {ari:.3f}")
print("Compare against Clay's ARI: 0.275 (README / paper Table 1).")

In [ ]:
contingency = pd.crosstab(
    pd.Series(clusters_matched, name="Prithvi embedding cluster"),
    pd.Series(class_names, name="WorldCover class"),
)

plt.figure(figsize=(10, 6))
plt.imshow(contingency.values, aspect="auto", cmap="viridis")
plt.xticks(range(len(contingency.columns)), contingency.columns, rotation=45, ha="right")
plt.yticks(range(len(contingency.index)), contingency.index)
plt.xlabel("ESA WorldCover class")
plt.ylabel("Prithvi embedding cluster")
plt.title("Prithvi embedding clusters vs. ground-truth land cover")
plt.colorbar(label="# chips")
plt.tight_layout()
plt.savefig("docs/figures/prithvi_cluster_vs_worldcover.png", dpi=150)
plt.show()
contingency

In [ ]:
print("\n=== Summary: Clay vs. Prithvi-EO-2.0 on the identical Front Range AOI ===")
print(f"{'metric':<32}{'Clay v1.5':<15}{'Prithvi-EO-2.0':<15}")
print(f"{'strongest PC vs. elevation (r)':<32}{'-0.82':<15}{best_r_elev:<15.3f}")
print(f"{'strongest PC vs. NDVI (r)':<32}{'-0.55':<15}{best_r_ndvi:<15.3f}")
print(f"{'ARI vs. WorldCover':<32}{'0.275':<15}{ari:<15.3f}")
print()
print("If Prithvi's numbers land in the same ballpark and same direction, that's real evidence")
print("the terrain-axis finding is a property of satellite foundation models generally, not an")
print("artifact of this one Clay checkpoint. If they don't, that's an equally real and useful")
print("result -- it would mean this project's claims should be scoped to Clay specifically.")
print()
print("Done. Commit the new docs/figures/prithvi_*.png files back to the repo.")
print("(This notebook doesn't touch docs/data/chips.geojson -- own independent grid/fetches.)")